<a href="https://colab.research.google.com/github/fezeusabrina/Telematics-driving-behavior./blob/main/Telematics_driving_behavior.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# Si vous avez mis le fichier à la racine de votre Drive :
chemin_fichier = '/content/drive/MyDrive/v2.csv'

print("Chargement des données en cours...")
# On garde l'astuce des 100 000 lignes pour coder vite ce soir !
df = pd.read_csv(chemin_fichier, nrows=100000)
print("Chargement terminé !\n")

print("--- INFOS SUR LES COLONNES ---")
print(df.info())

Chargement des données en cours...
Chargement terminé !

--- INFOS SUR LES COLONNES ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 17 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   tripID     100000 non-null  int64  
 1   deviceID   100000 non-null  float64
 2   timeStamp  100000 non-null  object 
 3   accData    100000 non-null  object 
 4   gps_speed  100000 non-null  float64
 5   battery    100000 non-null  float64
 6   cTemp      100000 non-null  float64
 7   dtc        100000 non-null  float64
 8   eLoad      100000 non-null  float64
 9   iat        100000 non-null  float64
 10  imap       100000 non-null  float64
 11  kpl        100000 non-null  float64
 12  maf        100000 non-null  float64
 13  rpm        100000 non-null  float64
 14  speed      100000 non-null  float64
 15  tAdv       100000 non-null  float64
 16  tPos       100000 non-null  float64
dtypes: float64(14), in

In [3]:
import numpy as np

# On sélectionne les capteurs intéressants
colonnes_utiles = ['speed', 'rpm', 'eLoad', 'tPos', 'cTemp', 'battery']
df_clean = df[colonnes_utiles].copy()

# On supprime les lignes vides au cas où
df_clean = df_clean.dropna()

# Notre petite règle pour étiqueter les données
def label_driving_style(row):
    if row['speed'] > 110 and row['rpm'] > 3200:
        return 'Agressif'
    elif row['rpm'] < 1800:
        return 'Eco'
    else:
        return 'Normal'

print("Calcul des styles de conduite en cours...")
df_clean['Target'] = df_clean.apply(label_driving_style, axis=1)

print("Terminé ! Voici la répartition :")
print(df_clean['Target'].value_counts())

Calcul des styles de conduite en cours...
Terminé ! Voici la répartition :
Target
Eco         75619
Normal      24360
Agressif       21
Name: count, dtype: int64


## Cellule 4 : Préparation pour le Machine Learning

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# X : Nos capteurs (on enlève la colonne 'Target' qu'on veut deviner)
X = df_clean.drop('Target', axis=1)
# y : Notre cible
y = df_clean['Target']

# Découpage 80% Entraînement / 20% Test
# Le fameux stratify=y permet de garder la même proportion de chaque classe !
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalisation : on met tout à la même échelle
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Données prêtes et normalisées ! Prêt pour l'entraînement.")

✅ Données prêtes et normalisées ! Prêt pour l'entraînement.


## Cellule 5 : Entraînement du Random Forest (Le test ultime)texte en gras

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print(" Entraînement du Random Forest en cours...")
# On crée le modèle
modele = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
# On l'entraîne sur nos données
modele.fit(X_train_scaled, y_train)

# On lui fait passer l'examen sur les 20% de données cachées
print(" Génération des prédictions...")
y_pred = modele.predict(X_test_scaled)

# Les résultats
print("\n--- 🏁 RÉSULTATS DU MODÈLE ---")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))

print("\nRapport de Précision :")
print(classification_report(y_test, y_pred))

 Entraînement du Random Forest en cours...
 Génération des prédictions...

--- 🏁 RÉSULTATS DU MODÈLE ---

Matrice de confusion :
[[    4     0     0]
 [    0 15124     0]
 [    0     0  4872]]

Rapport de Précision :
              precision    recall  f1-score   support

    Agressif       1.00      1.00      1.00         4
         Eco       1.00      1.00      1.00     15124
      Normal       1.00      1.00      1.00      4872

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



## Cellule 4 (Version "Anti-Triche")


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# X : On supprime la Target, MAIS AUSSI 'speed' et 'rpm' pour empêcher l'IA de tricher !
colonnes_a_supprimer = ['Target', 'speed', 'rpm']
X = df_clean.drop(colonnes_a_supprimer, axis=1)

# y : Notre cible reste la même
y = df_clean['Target']

# On affiche ce qui reste pour l'IA
print("L'IA va s'entraîner uniquement sur ces colonnes :", list(X.columns))

# Découpage 80% / 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Données sans fuite prêtes pour le vrai test !")

L'IA va s'entraîner uniquement sur ces colonnes : ['eLoad', 'tPos', 'cTemp', 'battery']
✅ Données sans fuite prêtes pour le vrai test !


## Cellule 5 : Le vrai Entraînement

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print(" Entraînement du modèle (sans triche) en cours...")
modele = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
modele.fit(X_train_scaled, y_train)

print("Génération des prédictions...")
y_pred = modele.predict(X_test_scaled)

print("\n--- 🏁 LES VRAIS RÉSULTATS DU MODÈLE ---")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))

print("\nRapport de Précision :")
print(classification_report(y_test, y_pred))

 Entraînement du modèle (sans triche) en cours...
Génération des prédictions...

--- 🏁 LES VRAIS RÉSULTATS DU MODÈLE ---

Matrice de confusion :
[[    0     2     2]
 [    2 13966  1156]
 [    1  3111  1760]]

Rapport de Précision :
              precision    recall  f1-score   support

    Agressif       0.00      0.00      0.00         4
         Eco       0.82      0.92      0.87     15124
      Normal       0.60      0.36      0.45      4872

    accuracy                           0.79     20000
   macro avg       0.47      0.43      0.44     20000
weighted avg       0.77      0.79      0.77     20000



## Cellule 5 (Version "Punition")

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print(" Entraînement du modèle (avec pénalité sur les classes rares)...")

# --- LA MAGIE EST ICI : class_weight='balanced' ---
modele_force = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
# --------------------------------------------------

modele_force.fit(X_train_scaled, y_train)

print("Génération des prédictions...")
y_pred_force = modele_force.predict(X_test_scaled)

print("\n--- RÉSULTATS AVEC CLASS_WEIGHT ---")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred_force))

print("\nRapport de Précision :")
print(classification_report(y_test, y_pred_force))

 Entraînement du modèle (avec pénalité sur les classes rares)...
Génération des prédictions...

--- RÉSULTATS AVEC CLASS_WEIGHT ---

Matrice de confusion :
[[    2     1     1]
 [   10 10276  4838]
 [   10   990  3872]]

Rapport de Précision :
              precision    recall  f1-score   support

    Agressif       0.09      0.50      0.15         4
         Eco       0.91      0.68      0.78     15124
      Normal       0.44      0.79      0.57      4872

    accuracy                           0.71     20000
   macro avg       0.48      0.66      0.50     20000
weighted avg       0.80      0.71      0.73     20000



## Transformer les données en "Film" (Séries Temporelles)

In [9]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print("🔄 Préparation des données pour le Deep Learning...")

# 1. On transforme nos textes ('Eco', 'Normal', 'Agressif') en numéros (0, 1, 2) pour l'IA
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(df_clean['Target'])

# 2. On garde les mêmes capteurs qu'hier (SANS tricher avec la vitesse !)
colonnes_a_supprimer = ['Target', 'speed', 'rpm']
X_dl = df_clean.drop(colonnes_a_supprimer, axis=1)

# Normalisation
scaler_dl = StandardScaler()
X_scaled_dl = scaler_dl.fit_transform(X_dl)

# 3. LA MAGIE DU TEMPS : On crée des fenêtres glissantes de 5 "secondes" (TIME_STEPS)
def create_sequences(data, labels, time_steps=5):
    Xs, ys = [], []
    for i in range(len(data) - time_steps):
        Xs.append(data[i:(i + time_steps)])
        ys.append(labels[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 5
X_seq, y_seq = create_sequences(X_scaled_dl, y_encoded, TIME_STEPS)

print(f"Format des données 3D pour le réseau de neurones : {X_seq.shape}")

# 4. On découpe en Train / Test
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq
)

# 5. On calcule la punition (class_weight) pour forcer l'IA à trouver les cas agressifs
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_dl), y=y_train_dl)
class_weights_dict = dict(enumerate(weights))

print("✅ Données séquentielles prêtes !")

🔄 Préparation des données pour le Deep Learning...
Format des données 3D pour le réseau de neurones : (99995, 5, 4)
✅ Données séquentielles prêtes !


## Construire le "Cerveau" (Architecture du Modèle)

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

print("Construction de l'architecture du réseau LSTM...")

# Initialisation du modèle
model = Sequential()

# Couche LSTM (La mémoire du modèle) - 64 neurones
model.add(LSTM(64, input_shape=(X_train_dl.shape[1], X_train_dl.shape[2])))

# Couche Dropout (On désactive 20% des neurones au hasard pour empêcher l'IA d'apprendre par cœur)
model.add(Dropout(0.2))

# Couche de réflexion (Dense)
model.add(Dense(32, activation='relu'))

# Couche de sortie : 3 neurones car on a 3 classes (Eco, Normal, Agressif)
model.add(Dense(3, activation='softmax'))

# Compilation du modèle (comment il apprend et corrige ses erreurs)
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary() # Affiche le plan de votre réseau

Construction de l'architecture du réseau LSTM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,843 (77.51 KB)

 Trainable params: 19,843 (77.51 KB)

 Non-trainable params: 0 (0.00 B)

## L'Entraînement (Le moment de vérité)

In [11]:
print(" Début de l'entraînement sous stéroïdes (Deep Learning)...")

history = model.fit(
    X_train_dl, y_train_dl,
    epochs=10, # Le modèle va lire le dataset 10 fois
    batch_size=32, # Il lit par paquet de 32 exemples
    validation_split=0.2, # Il se teste lui-même à chaque époque
    class_weight=class_weights_dict # On maintient la pénalité pour les classes rares !
)

print("✅ Entraînement terminé !")

 Début de l'entraînement sous stéroïdes (Deep Learning)...
Epoch 1/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - accuracy: 0.6645 - loss: 1.2213 - val_accuracy: 0.5200 - val_loss: 0.8392
Epoch 2/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 17s 7ms/step - accuracy: 0.6127 - loss: 0.8142 - val_accuracy: 0.6354 - val_loss: 0.6349
Epoch 3/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.6026 - loss: 0.7462 - val_accuracy: 0.5925 - val_loss: 0.6781
Epoch 4/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - accuracy: 0.5964 - loss: 0.6571 - val_accuracy: 0.6018 - val_loss: 0.6539
Epoch 5/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.6204 - loss: 0.7082 - val_accuracy: 0.5515 - val_loss: 0.9118
Epoch 6/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.6142 - loss: 0.5586 - val_accuracy: 0.6561 - val_loss: 0.6323
Epoch 7/10
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.6084 - loss: 0.6306 - val_accuracy: 0.5668 - val_loss: 0.6736
Epoch 8/10
2000/2000 ━━━

In [12]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

print("🔍 Génération des prédictions du LSTM sur les données cachées...")
# Le modèle calcule les probabilités pour chaque ligne
probabilites = model.predict(X_test_dl)

# On sélectionne la classe avec la plus haute probabilité
y_pred_dl = np.argmax(probabilites, axis=1)

print("\n--- 🏁 RÉSULTATS DU DEEP LEARNING (LSTM) ---")
print("\nMatrice de confusion :")
print(confusion_matrix(y_test_dl, y_pred_dl))

# On récupère nos mots ('Eco', 'Normal', 'Agressif') pour que ce soit lisible
noms_classes = encoder.inverse_transform(np.unique(y_test_dl))

print("\nRapport de Précision :")
# On affiche le rapport final !
print(classification_report(y_test_dl, y_pred_dl, target_names=noms_classes))

🔍 Génération des prédictions du LSTM sur les données cachées...
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

--- 🏁 RÉSULTATS DU DEEP LEARNING (LSTM) ---

Matrice de confusion :
[[   4    0    0]
 [ 721 8991 5411]
 [ 816  909 3147]]

Rapport de Précision :
              precision    recall  f1-score   support

    Agressif       0.00      1.00      0.01         4
         Eco       0.91      0.59      0.72     15123
      Normal       0.37      0.65      0.47      4872

    accuracy                           0.61     19999
   macro avg       0.43      0.75      0.40     19999
weighted avg       0.78      0.61      0.66     19999

